# HW1: GPU Roofline Model

**Setup:**
1. `Runtime > Change runtime type > GPU`
2. Upload your `.py` files (next cell), or clone your repo
3. Run cells top to bottom

In [1]:
# Check GPU
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__}")

GPU: Tesla T4
PyTorch: 2.11.0+cu128


In [2]:
# Option A: Upload files from your machine
from google.colab import files
uploaded = files.upload()  # Select: hw1_task_impl.py, hw1_runtime.py, hw1_task.py

KeyboardInterrupt: 

In [ ]:
# Option B (alternative): Clone your repo instead of uploading
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git
# %cd YOUR_REPO/hw1

In [ ]:
# Patch hw1_runtime.py to support Colab GPUs (T4, A100, V100)
import re

with open("hw1_runtime.py", "r") as f:
    content = f.read()

# Add T4/A100/V100 specs if not already present
if '"T4"' not in content:
    extra_gpus = '''
    "T4": {
        "label": "NVIDIA Tesla T4 16GB GDDR6",
        "peak_flops": 8.1e12,
        "peak_bw": 320e9,
    },
    "A100": {
        "label": "NVIDIA A100 80GB HBM2e",
        "peak_flops": 19.5e12,
        "peak_bw": 2.0e12,
    },
    "V100": {
        "label": "NVIDIA V100 16GB HBM2",
        "peak_flops": 15.7e12,
        "peak_bw": 900e9,
    },
'''
    content = content.replace(
        'GPU_SPECS = {',
        'GPU_SPECS = {' + extra_gpus,
        1
    )
    with open("hw1_runtime.py", "w") as f:
        f.write(content)
    print("Patched hw1_runtime.py with T4/A100/V100 specs")
else:
    print("GPU specs already present, no patch needed")

In [ ]:
# Run the homework
import importlib, hw1_runtime, hw1_task_impl
importlib.reload(hw1_runtime)
importlib.reload(hw1_task_impl)

from hw1_runtime import measure_roofline_points, plot_roofline, print_header, save_roofline_data
from hw1_task_impl import benchmark_fn, compute_elementwise_metrics, lowest_ai_fn, make_compute_fn

print_header()
print("Running benchmarks...")
results = measure_roofline_points(
    lowest_ai_fn=lowest_ai_fn,
    make_compute_fn=make_compute_fn,
    benchmark_fn=benchmark_fn,
    compute_elementwise_metrics=compute_elementwise_metrics,
)
save_roofline_data(results)
print("\nGenerating plot...")
plot_roofline(results)

In [ ]:
# Display the roofline plot inline
from IPython.display import Image
Image("results/roofline.png")